In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q scikit-learn
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
!pip install -q -U transformers accelerate datasets
!pip install -q einops
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 30.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 38.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into a

In [2]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm 
import warnings
import urllib.request
import zipfile

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. VÁ LỖI KIẾN TRÚC INTERNVL & HUGGINGFACE
# ==========================================
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cudnn.benchmark = True 

runName = "Centralized_SLAKE_InternVL3.5_LoRA_E2_B8_FP16"
kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "slake")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_centralized_model.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ
# ==========================================
def clearMemory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def getPeakReservedMemory():
    return f"{torch.cuda.memory_reserved() / (1024 ** 3):.2f} GB" if torch.cuda.is_available() else "0.00 GB"

def processImage(pilImg, targetSize=448):
    if pilImg.mode != "RGB":
        pilImg = pilImg.convert("RGB")
    imageTransform = T.Compose([
        T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])
    return imageTransform(pilImg).unsqueeze(0)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH (INTERNVL3.5-2B)
# ==========================================
print("--- KHỞI TẠO MÔ HÌNH (INTERNVL3.5-2B, LORA 448x448, FP16 TENSOR CORES) ---")
clearMemory()

modelId = "OpenGVLab/InternVL3_5-2B-Instruct"
baseModel = AutoModel.from_pretrained(
    modelId,
    torch_dtype=torch.float16, 
    trust_remote_code=True,
    low_cpu_mem_usage=False
).cuda()

tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
baseModel.config.use_cache = False
baseModel.gradient_checkpointing_enable()

loraConfig = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

globalPeftModel = get_peft_model(baseModel, loraConfig)

trainableParams = sum(p.numel() for p in globalPeftModel.parameters() if p.requires_grad)
totalParams = sum(p.numel() for p in globalPeftModel.parameters())
trainPct = 100 * trainableParams / totalParams
print(f"[Metrics] Trainable Params: {trainableParams:,} (~{trainPct:.3f}%)")

# ==========================================
# 3. TRÍCH XUẤT ĐỘNG ID TOKEN HÌNH ẢNH
# ==========================================
print(" Đang trích xuất tự động Image Context ID từ lõi mô hình...")
interceptedIds = None
originalForward = globalPeftModel.forward

def hookForward(*args, **kwargs):
    global interceptedIds
    if 'input_ids' in kwargs and kwargs['input_ids'] is not None:
        interceptedIds = kwargs['input_ids'].detach().cpu()
    elif len(args) > 1 and args[1] is not None:
        interceptedIds = args[1].detach().cpu()
    raise RuntimeError("STOP_INTERCEPT")

globalPeftModel.forward = hookForward
try:
    dummyPx = torch.zeros((1, 3, 448, 448), dtype=torch.float16).cuda()
    globalPeftModel.chat(tokenizer, dummyPx, "<image>", generation_config=dict(max_new_tokens=1))
except RuntimeError as e:
    if str(e) != "STOP_INTERCEPT":
        pass
except Exception:
    pass
finally:
    globalPeftModel.forward = originalForward

if interceptedIds is not None:
    counts = torch.bincount(interceptedIds.view(-1))
    imgContextId = int(counts.argmax().item())
else:
    imgContextId = getattr(globalPeftModel.base_model.model, 'img_context_token_id', tokenizer.convert_tokens_to_ids('<IMG_CONTEXT>'))

print(f"✅ Đã tìm thấy Image Context ID an toàn: {imgContextId}")

# ==========================================
# 4. CHUẨN BỊ DỮ LIỆU SLAKE (TẬP TRUNG)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU SLAKE ENGLISH ---")

if not os.path.exists("./imgs"):
    print(" [Hệ thống] Đang tải file ảnh imgs.zip từ Hugging Face...")
    urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
    with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
        zipRef.extractall("./")
    os.remove("imgs.zip")
    print(" [Hệ thống] Đã chuẩn bị xong thư mục ảnh cục bộ!")

def loadSlakeImage(imgName):
    imgPath = os.path.join(".", "imgs", imgName.replace("imgs/", ""))
    return Image.open(imgPath).convert("RGB")

dataset = load_dataset("BoKelvin/SLAKE", cache_dir=dataDir)

englishTrain = dataset["train"].filter(lambda x: x["q_lang"] == "en").shuffle(seed=42)
englishTestAll = dataset["test"].filter(lambda x: x["q_lang"] == "en")
testDataset = englishTestAll.shuffle(seed=42).select(range(min(500, len(englishTestAll))))

print(f"Tổng số mẫu tập Train: {len(englishTrain)}")
print(f"Tổng số mẫu tập Test : {len(testDataset)}")

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\nStrictly follow these rules:\n"
    "1. Analyze the provided image.\n2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    pixelValuesList, inputIdsList, labelsList = [], [], []
    for item in batch:
        img = loadSlakeImage(item["img_name"])
        q, a = item["question"], str(item["answer"]).strip()

        promptPart1 = f"{systemInstruction}\nUser: <img>"
        promptPart2 = f"</img>\nQuestion: {q}\nFinal Answer:\nAssistant: "

        part1Ids = tokenizer(promptPart1, return_tensors="pt", add_special_tokens=True).input_ids[0]
        part2Ids = tokenizer(promptPart2, return_tensors="pt", add_special_tokens=False).input_ids[0]
        answerIds = tokenizer(a + getattr(tokenizer, 'eos_token', ''), return_tensors="pt", add_special_tokens=False).input_ids[0]

        imgTokens = torch.full((256,), imgContextId, dtype=torch.long)
        inputIds = torch.cat([part1Ids, imgTokens, part2Ids, answerIds], dim=0)

        lbls = inputIds.clone()
        promptLen = len(part1Ids) + 256 + len(part2Ids)
        lbls[:promptLen] = -100 

        inputIdsList.append(inputIds)
        labelsList.append(lbls)
        pixelValuesList.append(processImage(img))

    maxLen = max(len(ids) for ids in inputIdsList)
    padId = getattr(tokenizer, 'pad_token_id', getattr(tokenizer, 'eos_token_id', 0))
    if padId is None:
        padId = 0

    paddedInputIds, paddedLabels, attentionMask = [], [], []
    for ids, lbls in zip(inputIdsList, labelsList):
        padLen = maxLen - len(ids)
        paddedInputIds.append(torch.cat([ids, torch.full((padLen,), padId, dtype=torch.long)]))
        paddedLabels.append(torch.cat([lbls, torch.full((padLen,), -100, dtype=torch.long)]))
        attentionMask.append(torch.cat([torch.ones(len(ids), dtype=torch.long), torch.zeros(padLen, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(paddedInputIds),
        "labels": torch.stack(paddedLabels),
        "attention_mask": torch.stack(attentionMask),
        "pixel_values": torch.cat(pixelValuesList, dim=0).to(torch.float16),
        "image_flags": torch.ones((len(batch), 1), dtype=torch.long) 
    }

# ÉP XUNG VRAM: Nâng Batch Size lên 8
batchSize = 8
trainLoader = DataLoader(englishTrain, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=0, pin_memory=True)

# ==========================================
# 5. HÀM ĐÁNH GIÁ (ĐÃ TỐI ƯU CLOSED QUESTIONS)
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateModel(model, datasetInput):
    model.eval()
    print("\n [Đánh giá] Đang tiến hành Test...")
    openPreds, openRefs, closePreds, closeRefs, inferenceTimes = [], [], [], [], []
    overallCorrect = 0
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)

    for item in evalLoop:
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "":
            ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"

        promptText = f"{systemInstruction}\n[Target Case Image]\n<image>\n\nCurrent Question: {q}\nBased on the visual evidence, what is the correct answer?\nFinal Answer: "
        img = loadSlakeImage(item["img_name"])
        pixelValues = processImage(img).to(torch.float16).cuda()

        startTime = time.time()
        with torch.inference_mode():
            aiAnswerRaw = model.chat(
                tokenizer=tokenizer,
                pixel_values=pixelValues,
                question=promptText,
                generation_config=dict(max_new_tokens=15, do_sample=False)
            )
        inferenceTimes.append(time.time() - startTime)
        
        aiAnswer = aiAnswerRaw.lower().replace("final answer:", "").strip()
        aiAnswerClean = cleanOpenAnswer(aiAnswer)
        gtAnswerClean = cleanOpenAnswer(gtAnswer)

        if ansType == "CLOSED":
            if gtAnswer in ['yes', 'no']:
                hasYes = any(word in aiAnswer for word in yesKeywords)
                hasNo = any(word in aiAnswer for word in noKeywords)
                
                if hasYes and not hasNo:
                    predClean = 'yes'
                elif hasNo and not hasYes:
                    predClean = 'no'
                else:
                    firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                    firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                    predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
                targetGt = gtAnswer
            else:
                predClean = aiAnswerClean if aiAnswerClean else "unknown"
                targetGt = gtAnswerClean if gtAnswerClean else gtAnswer
                
            closePreds.append(predClean)
            closeRefs.append(targetGt)
            
            if predClean == targetGt:
                overallCorrect += 1
        else:
            predOpen = aiAnswerClean if aiAnswerClean else "none"
            refOpen = gtAnswerClean if gtAnswerClean else "none"
            openPreds.append(predOpen)
            openRefs.append(refOpen)
            
            if predOpen == refOpen:
                overallCorrect += 1

        del pixelValues
        torch.cuda.empty_cache()

    res = {
        "avgInferenceTime": np.mean(inferenceTimes) if inferenceTimes else 0,
        "overallAccuracy": overallCorrect / len(datasetInput) if len(datasetInput) > 0 else 0
    }
    if closeRefs:
        res["Closed"] = {
            "ACC": accuracy_score(closeRefs, closePreds),
            "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0),
            "count": len(closeRefs)
        }
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {
            "BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"],
            "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"],
            "BERTScore": np.mean(bertRes["f1"]),
            "count": len(openRefs)
        }
    gc.collect()
    return res

# ==========================================
# 6. VÒNG LẶP HUẤN LUYỆN TẬP TRUNG (2 EPOCHS)
# ==========================================
totalEpochs = 2  # Đã cập nhật còn 2 Epochs
learningRate = 1e-4
optimizer = torch.optim.AdamW(globalPeftModel.parameters(), lr=learningRate, weight_decay=0.01)
accumulationSteps = 1
scheduler = CosineAnnealingLR(optimizer, T_max=totalEpochs * len(trainLoader) // accumulationSteps)

performanceLog = []
peakVramRecorded = "0.00 GB"
print(f"\n=== BẮT ĐẦU HUẤN LUYỆN TẬP TRUNG ({totalEpochs} EPOCHS, BATCH {batchSize}) ===")

for epoch in range(1, totalEpochs + 1):
    globalPeftModel.train()
    startTime = time.time()
    totalLoss = 0

    progressLoop = tqdm(trainLoader, desc=f"Epoch {epoch}/{totalEpochs}", leave=True, dynamic_ncols=True)

    for step, batch in enumerate(progressLoop):
        batch = {k: v.cuda(non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss = globalPeftModel(**batch).loss / accumulationSteps

        loss.backward()

        if (step + 1) % accumulationSteps == 0 or (step + 1) == len(trainLoader):
            torch.nn.utils.clip_grad_norm_(globalPeftModel.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        stepLoss = loss.item() * accumulationSteps
        totalLoss += stepLoss
        progressLoop.set_postfix(Loss=f"{stepLoss:.4f}", LR=f"{scheduler.get_last_lr()[0]:.2e}")

        del batch, loss

    avgLoss = totalLoss / len(trainLoader)
    trainTime = time.time() - startTime
    peakVramRecorded = getPeakReservedMemory()

    print(f" Epoch {epoch} Xong | Avg Loss: {avgLoss:.4f} | VRAM: {peakVramRecorded} | Time: {trainTime:.2f}s")

    # Đánh giá sau mỗi Epoch
    evalRes = evaluateModel(globalPeftModel, testDataset)

    roundAccuracy = evalRes["overallAccuracy"] * 100
    roundAccClosed = evalRes.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = evalRes.get("Closed", {}).get("F1", 0) * 100
    roundBleu = evalRes.get("Open", {}).get("BLEU-1", 0)
    roundRouge = evalRes.get("Open", {}).get("ROUGE-L", 0)
    roundBert = evalRes.get("Open", {}).get("BERTScore", 0)

    roundLog = {
        "Epoch": epoch,
        "Avg Loss": round(avgLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "Avg Train Time (s)": round(trainTime, 2)
    }
    performanceLog.append(roundLog)

    print(f" -> Acc: {roundAccuracy:.2f}% | Closed Acc: {roundAccClosed:.2f}%, F1: {roundF1Closed:.2f}% | BLEU-1: {roundBleu:.4f}, ROUGE-L: {roundRouge:.4f}, BERT: {roundBert:.4f}")

    pd.DataFrame(performanceLog).to_csv(logPath, index=False)

    # Lưu trọng số
    weights = {k: v.cpu().clone() for k, v in globalPeftModel.state_dict().items() if "lora" in k}
    torch.save(weights, checkpointPath)

# ==========================================
# 7. BÁO CÁO TỔNG KẾT CHI TIẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: CENTRALIZED INTERNVL3.5 (SLAKE, LORA 448x448, BATCH {batchSize}, FP16, {totalEpochs} EPOCHS) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~{trainPct:.3f}%)")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {evalRes['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL EPOCH CUỐI)")
if "Closed" in evalRes:
    print(f"  - CLOSED QUESTIONS ({evalRes['Closed']['count']} samples):")
    print(f"    * Accuracy : {evalRes['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalRes['Closed']['F1']*100:.2f}%")
if "Open" in evalRes:
    print(f"  - OPEN QUESTIONS ({evalRes['Open']['count']} samples):")
    print(f"    * BLEU-1   : {evalRes['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalRes['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalRes['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
commTable = pd.DataFrame(performanceLog)
print(commTable[['Epoch', 'Avg Loss', 'Overall Acc (%)', 'Closed Acc (%)', 'Open BLEU-1', 'Avg Train Time (s)']].to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO MÔ HÌNH (INTERNVL3.5-2B, LORA 448x448, FP16 TENSOR CORES) ---


config.json:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

configuration_internvl_chat.py:   0%|          | 0.00/4.70k [00:00<?, ?B/s]

configuration_intern_vit.py:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

modeling_internvl_chat.py:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

... (36 outputs left out)

 Epoch 2 Xong | Avg Loss: 0.1558 | VRAM: 11.76 GB | Time: 3614.03s

 [Đánh giá] Đang tiến hành Test...


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

 -> Acc: 83.20% | Closed Acc: 87.57%, F1: 86.65% | BLEU-1: 0.7912, ROUGE-L: 0.8519, BERT: 0.9810

 FINAL SUMMARY REPORT: CENTRALIZED INTERNVL3.5 (SLAKE, LORA 448x448, BATCH 8, FP16, 2 EPOCHS) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 34,865,152 (~1.463%)
  + Peak Reserved Memory   : 11.76 GB
  + Average Inference Time : 0.3673 giây/ảnh

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL EPOCH CUỐI)
  - CLOSED QUESTIONS (177 samples):
    * Accuracy : 87.57%
    * F1-Score : 86.65%
  - OPEN QUESTIONS (323 samples):
    * BLEU-1   : 0.7912
    * ROUGE-L  : 0.8519
    * BERTScore: 0.9810

[3] CONVERGENCE LOG CHI TIẾT
 Epoch  Avg Loss  Overall Acc (%)  Closed Acc (%)  Open BLEU-1  Avg Train Time (s)
     1    0.4063             80.2           88.70       0.7322             3626.84
     2    0.1558             83.2           87.57       0.7912             3614.03

 QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG 
